In [1]:
from pathlib import Path
import os
import sys

# Chercher la racine du projet contenant src/config.py
candidates = [Path.cwd(), *Path.cwd().parents, Path("/workspace/NeuroVision_AI")]

ROOT = next(
    (p.resolve() for p in candidates if (p / "src" / "config.py").is_file()),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        "Projet introuvable : impossible de trouver src/config.py. "
        f"Répertoire actuel : {Path.cwd()}"
    )

# Se placer à la racine du projet
os.chdir(ROOT)

# Ajouter la racine au chemin de recherche Python
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Racine du projet :", ROOT)
print("Répertoire actuel :", Path.cwd())
print("Chemin Python :", sys.path[0])
print("Dossier src présent :", (ROOT / "src").is_dir())

Racine du projet : /workspace/NeuroVision_AI
Répertoire actuel : /workspace/NeuroVision_AI
Chemin Python : /workspace/NeuroVision_AI
Dossier src présent : True


In [2]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from src.config import IMAGE_SIZE, LABELS_BY_FOLDER
from src.data.loading import charger_dataset_path,verifier_chargement, preparer_data, print_class_counts
from src.data.splitting import spliting_data
from src.data.pipeline import preprocessing, preprocessing_v2
from src.evaluation.metrics import evaluate_and_save_metrics
from src.evaluation.plots import plot_learning_curves, plot_confusion_matrix, save_example_prediction
from src.config import OUTPUT_DIR, MODEL_DIR, RANDOM_STATE
from src.training.train_transfer_learning import train_transfer_learning


print("✅ Imports réussis")
print("Taille des images :", IMAGE_SIZE)
print("Classes :", LABELS_BY_FOLDER)

I0000 00:00:1792792321.362216   16751 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


✅ Imports réussis
Taille des images : (224, 224)
Classes : {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


In [3]:
import importlib
import inspect

module = importlib.import_module(
    "src.training.train_transfer_learning"
)

module = importlib.reload(module)

print("Fichier réellement utilisé :")
print(module.__file__)

print("\nFonctions définies dans ce module :")
for nom, objet in vars(module).items():
    if inspect.isfunction(objet) and objet.__module__ == module.__name__:
        print(nom, inspect.signature(objet))

Fichier réellement utilisé :
/workspace/NeuroVision_AI/src/training/train_transfer_learning.py

Fonctions définies dans ce module :
train_transfer_learning (Train_ds, Val_ds)


In [4]:
dataset_path = charger_dataset_path()
#Verification
images, labels = verifier_chargement(dataset_path)
#Transformation à 2d arrays
images_array_2d, labeles_array_2d = preparer_data(images, labels, IMAGE_SIZE)

Dataset déjà présent dans le cache.
Chemin : /root/.cache/kagglehub/datasets/devana2004/dataset/versions/1/Data
Contenu : ['glioma', 'meningioma', 'notumor', 'pituitary']
Path : /root/.cache/kagglehub/datasets/devana2004/dataset/versions/1/Data


: 

: 

: 

In [4]:

dataset_path = charger_dataset_path()
#Verification
images, labels = verifier_chargement(dataset_path)
#Transformation à 2d arrays
images_array_2d, labeles_array_2d = preparer_data(images, labels, IMAGE_SIZE)
# Aperçu des premières images : les images ont déjà été converties en RGB.
plt.figure(figsize=(10, 10))
n_images = min(9, len(images))
for i in range(n_images):
      plt.subplot(3, 3, i + 1)
      plt.imshow(images[i])
      nom_class = next(
            (k for k, v in LABELS_BY_FOLDER.items() if v == labels[i]), None
      )
      plt.title(nom_class)
plt.tight_layout()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(OUTPUT_DIR / "example1.png")
plt.close()
noms_classes, nombres_images = print_class_counts(labels)
plt.figure()
plt.bar(noms_classes, nombres_images)
plt.title("Nombre d'images par classe")
plt.xlabel("Classes")
plt.ylabel("Nombre d'images")
plt.savefig(OUTPUT_DIR / "image_par_class.png")
plt.close()
X_train, X_test, X_val, y_train, y_test, y_val = spliting_data(
      images_array_2d, labeles_array_2d, RANDOM_STATE
)
print("X_train :", type(X_train))
print("Shape :", X_train.shape)
print("Type des données :", X_train.dtype)
print("Taille en mémoire :", round(X_train.nbytes / 1024**3, 2), "GB")
print("y_train :", np.asarray(y_train).shape)
print("GPU détecté :", tf.config.list_physical_devices("GPU"))
Train_ds = preprocessing_v2(X_train, y_train, training=True)
Test_ds = preprocessing_v2(X_test, y_test, training=False)
Val_ds = preprocessing_v2(X_val, y_val, training=False)
for image_batch, label_batch in Train_ds.take(1):
      print(image_batch.shape)
      print(label_batch.shape)
for batch_images, batch_labels in Train_ds.take(1):
      print("Forme des images :", batch_images.shape)
      print("Valeur minimale :", float(tf.reduce_min(batch_images)))
      print("Valeur maximale :", float(tf.reduce_max(batch_images)))
      print("Labels :", batch_labels[:10].numpy())
model, history = train_transfer_learning(Train_ds, Val_ds)
plot_learning_curves(history)
test_loss, test_acc = model.evaluate(Test_ds, verbose=2)
print("Test loss :", test_loss)
print("Test accuracy :", test_acc)
y_prob = model.predict(Test_ds)
y_pred = np.argmax(y_prob, axis=1)
y_true = np.concatenate([batch_labels.numpy() for _, batch_labels in Test_ds])
plot_confusion_matrix(y_true, y_pred)
evaluate_and_save_metrics(model, Test_ds, history)
images_all = np.concatenate([batch_images.numpy() for batch_images, _ in Test_ds])
save_example_prediction(images_all, y_true, y_pred)
best_model_path = MODEL_DIR / "best_model.keras"
best_model = tf.keras.models.load_model(best_model_path)
best_model.summary()
best_model.evaluate(Test_ds)


Dataset déjà présent dans le cache.
Chemin : /root/.cache/kagglehub/datasets/devana2004/dataset/versions/1/Data
Contenu : ['glioma', 'meningioma', 'notumor', 'pituitary']
Path : /root/.cache/kagglehub/datasets/devana2004/dataset/versions/1/Data


: 

: 

: 